# Train a spam message classifier with Watson NLP
Trains a custom text classifier (watson_nlp GenericEnsemble, TFIDF-SVM base) on the UCI SMS Spam Collection so a trained model - not keyword rules - makes the prediction.

Honest framing: the dataset labels are **spam / ham** (not proven fraud / genuine). The model learns to detect spam-style messages. Evaluation uses a genuinely held-out test split plus fresh challenge messages never seen in training.

In [ ]:
# Cell 1: Download UCI SMS Spam Collection, verify counts, deduplicate
import urllib.request, zipfile, io, collections

url = 'https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip'
data = urllib.request.urlopen(url, timeout=90).read()
z = zipfile.ZipFile(io.BytesIO(data))
print('zip contents:', z.namelist())
raw = z.read('SMSSpamCollection').decode('utf-8', errors='replace')

rows = []
for line in raw.strip().split('\n'):
    parts = line.split('\t', 1)
    if len(parts) == 2:
        lab, txt = parts[0].strip().lower(), parts[1].strip()
        if lab in ('spam', 'ham') and txt:
            rows.append((lab, txt))

print('raw labelled rows:', len(rows))
print('label distribution (raw):', dict(collections.Counter(l for l, _ in rows)))

# Deduplicate exact (case-insensitive) message text BEFORE splitting, to prevent leakage
seen = set(); ded = []
for lab, txt in rows:
    k = ' '.join(txt.lower().split())
    if k not in seen:
        seen.add(k); ded.append((lab, txt))
print('rows after dedupe:', len(ded))
print('label distribution (deduped):', dict(collections.Counter(l for l, _ in ded)))

In [ ]:
# Cell 2: Stratified 80/20 split, write train_data.json (watson_nlp classification format)
import random, json
random.seed(42)
random.shuffle(ded)
spam = [r for r in ded if r[0] == 'spam']
ham = [r for r in ded if r[0] == 'ham']

def split80(lst):
    n = int(len(lst) * 0.8)
    return lst[:n], lst[n:]

spam_tr, spam_te = split80(spam)
ham_tr, ham_te = split80(ham)
train = spam_tr + ham_tr
test = spam_te + ham_te
random.shuffle(train); random.shuffle(test)

# safety check: no text overlap between train and test
tr_keys = {' '.join(t.lower().split()) for _, t in train}
overlap = [t for _, t in test if ' '.join(t.lower().split()) in tr_keys]
print('train/test overlap (must be 0):', len(overlap))

with open('train_data.json', 'w') as f:
    json.dump([{'text': t, 'labels': [l]} for l, t in train], f)
print('train:', len(train), 'test:', len(test))
print('train distribution:', dict(collections.Counter(l for l, _ in train)))
print('test distribution:', dict(collections.Counter(l for l, _ in test)))

In [ ]:
# Cell 3: Train GenericEnsemble classifier (TFIDF-SVM base) - documented watson_nlp path
import watson_nlp
from watson_nlp.workflows.classification.generic_ensemble import GenericEnsemble
from watson_nlp.workflows.classification.base_classifier import TFidfSvm

syntax_model = watson_nlp.load('syntax_izumo_en_stock')

classification_model = GenericEnsemble.train(
    'train_data.json',
    syntax_model,
    base_classifiers_params=[TFidfSvm.TrainParams(syntax_model=syntax_model)],
    weights=[1])
print('training complete')

In [ ]:
# Cell 4: Inspect prediction output shape, then evaluate on the held-out test set
sample = classification_model.run('Free entry in our weekly prize draw').to_dict()
print('sample prediction dict:', sample)

def extract_label(pred_dict):
    classes = pred_dict.get('classes', [])
    if not classes:
        return None, 0.0
    top = classes[0]
    label = top.get('label', top.get('class_name'))
    conf = top.get('confidence', top.get('score', 0.0))
    return label, conf

# per-document run for reliability on ~1k docs (TFIDF-SVM inference is fast)
y_true, y_pred, confs = [], [], []
for lab, txt in test:
    p, c = extract_label(classification_model.run(txt).to_dict())
    y_true.append(lab); y_pred.append(p); confs.append(c)

tp = sum(1 for a, b in zip(y_true, y_pred) if a == 'spam' and b == 'spam')
fp = sum(1 for a, b in zip(y_true, y_pred) if a == 'ham' and b == 'spam')
fn = sum(1 for a, b in zip(y_true, y_pred) if a == 'spam' and b == 'ham')
tn = sum(1 for a, b in zip(y_true, y_pred) if a == 'ham' and b == 'ham')
acc = (tp + tn) / len(y_true)
prec = tp / (tp + fp) if tp + fp else 0
rec = tp / (tp + fn) if tp + fn else 0
f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0
print(f'HELD-OUT TEST RESULTS (n={len(y_true)})')
print(f'accuracy:           {acc:.4f}')
print(f'spam precision:     {prec:.4f}')
print(f'spam recall:        {rec:.4f}')
print(f'spam F1:            {f1:.4f}')
print(f'confusion: tp={tp} fp={fp} fn={fn} tn={tn}')
# show a few false positives/negatives for honest inspection
print('\nSample false positives (ham predicted spam):')
for (lab, txt), p in list(zip(test, y_pred))[:0]: pass
shown = 0
for (lab, txt), p in zip(test, y_pred):
    if lab == 'ham' and p == 'spam' and shown < 5:
        print(' -', txt[:120]); shown += 1
print('Sample false negatives (spam predicted ham):')
shown = 0
for (lab, txt), p in zip(test, y_pred):
    if lab == 'spam' and p == 'ham' and shown < 5:
        print(' -', txt[:120]); shown += 1

In [ ]:
# Cell 5: Fresh challenge cases - written new for this test, NOT from the dataset
challenge = [
    ('spam-style', "URGENT: Your SBI account has been blocked. Verify your KYC immediately: http://sbi-kyc-update.example.com"),
    ('spam-style', "Congratulations! You've won a Rs.50,000 Amazon voucher. Claim your prize now at bit.ly/3xPrize"),
    ('spam-style', "Your package could not be delivered. Pay the Rs.20 redelivery fee here: http://india-post-fee.example.com"),
    ('spam-style', "Part-time job offer! Earn Rs.5000 per day from home. Pay Rs.499 registration fee to start."),
    ('spam-style', "Dear customer, your electricity will be disconnected tonight. Call 98XXXXXXXX immediately to pay pending bill."),
    ('ham-style', "Are we still meeting at the library at 4 today?"),
    ('ham-style', "Your OTP for login is 482913. Do not share it with anyone."),
    ('ham-style', "Reminder: parent-teacher meeting on Friday at 10 AM in the main hall."),
    ('ham-style', "Happy birthday da! Hope you have a great year ahead"),
    ('ham-style', "Can you send me the notes from yesterday's lecture?"),
]
print('FRESH CHALLENGE CASES (never in training or test):')
for note, txt in challenge:
    p, c = extract_label(classification_model.run(txt).to_dict())
    print(f'[{note:10s}] -> {p} (conf {c:.2f}) | {txt[:80]}')

In [ ]:
# Cell 6: Save the trained model to the project as an asset
classification_model.save('spam-message-classifier')
print('model saved as spam-message-classifier')
import os
print(os.listdir('spam-message-classifier') if os.path.isdir('spam-message-classifier') else 'saved (archive format)')

## Next steps (not run here)
- Optionally add USE-SVM or CNN base classifiers to the ensemble and compare quality.
- Deploy as a watsonx.ai Runtime online deployment (Python function) for live scoring.
- Stop the runtime when done to conserve CUH.

In [ ]:
# Cell 7: Package the model for durable storage (the runtime filesystem is ephemeral)
import tarfile, os
with tarfile.open('spam-message-classifier.tar.gz', 'w:gz') as tar:
    tar.add('spam-message-classifier')
print('archive size bytes:', os.path.getsize('spam-message-classifier.tar.gz'))
# Durable copy: the archive is retained privately by the project owner (this runtime has no
# project access token injected - ibm_watson_studio_lib project save returns 401). The model is
# NOT redistributed publicly: the dataset licence field is null in the UCI registry, so the
# trained asset stays private until redistribution terms are verified. Training is deterministic
# (seed 42), so the model can also be regenerated by re-running cells 1-6.

In [ ]:
# Cell 8: Reload verification - delete the model dir, restore ONLY from the archive, predict
import tarfile, shutil, os
if os.path.isdir('spam-message-classifier'):
    shutil.rmtree('spam-message-classifier')
with tarfile.open('spam-message-classifier.tar.gz') as tar:
    tar.extractall('.')
import watson_nlp
reloaded = watson_nlp.load('spam-message-classifier')
for msg in ["URGENT: Your SBI account has been blocked. Verify your KYC immediately: http://sbi-kyc-update.example.com",
            "Hey, are we still meeting for lunch tomorrow at 1?"]:
    print(reloaded.run(msg).to_dict())
print('RELOAD VERIFIED from tar.gz archive')

# On a FRESH runtime with no archive: re-run cells 1-6 (deterministic) or copy the privately
# retained spam-message-classifier.tar.gz into the working directory first.